# 00 - Data and preprocessing

Every other notebook in this series starts from the model-ready tables built here.

**What you'll learn**
- How each raw dataset becomes a numeric, model-ready table (one small step at a time).
- What "privileged" and "unprivileged" mean in callmefair, and how each group is encoded.
- How to check that our preprocessing reproduces the CSVs shipped in `examples/data/`.
- How callmefair receives the groups (`privileged_groups` / `unprivileged_groups`).

Runtime: a few seconds. Raw files are optional (see `examples/data/raw/README.md`); without them the notebook describes the shipped CSVs instead.

In [1]:
from fhg_examples import setup
setup()

import pandas as pd
from fhg_examples.datasets import RAW_DIR, SPECS, load, load_raw, group_summary
from fhg_examples import preprocessing as pp

def raw_available(name):
    return (RAW_DIR / SPECS[name].raw_file).exists()

## Two conventions to know first

**Privileged / unprivileged.** In fairness toolkits (AIF360, callmefair), the *privileged group* is simply the **reference group**, the one whose outcomes other groups are compared against. It is encoded as `1` in the sensitive column; the *unprivileged group* is encoded `0`. This is a **modelling choice, not a moral claim**: it says "measure disparity relative to this group", usually because the literature documents that the other group is disadvantaged in the domain. Which group is the reference is a decision made per attribute, and it deserves scrutiny; each dataset below shows the group sizes and base rates so you can judge it.

**Favorable label.** callmefair uses `favorable_label = 1`. A positive prediction is therefore the *favorable* outcome, even when the label itself is "bad news" (a readmission, a stroke). Read it as "flagged for follow-up care or resources": fairness then asks whether such flags are handed out equitably across groups.

**Sensitive attributes** are kept as ordinary 0/1 columns in the data, so a model may see them. Mitigation methods in later notebooks decide how to treat them.

## 1. Diabetes: hospital readmission

*Source:* UCI ML Repository #296, "Diabetes 130-US Hospitals for Years 1999-2008" (Strack et al., BioMed Research International, 2014), license CC BY 4.0. One row per hospital encounter of a diabetic patient.

*Label:* `readmitted` = 1 if the patient came back **within 30 days**.

In [2]:
spec = SPECS["diabetes"]
HAS_RAW = raw_available("diabetes")
if HAS_RAW:
    raw = load_raw("diabetes")
    print(raw.shape)
    display(raw[["race", "gender", "age", "time_in_hospital", "A1Cresult", "metformin", "readmitted"]].head(4))
else:
    print("Raw file missing: download it as described in examples/data/raw/README.md.")
    print("Showing the shipped processed CSV instead.")
    display(load("diabetes").iloc[:4, :8])

(101766, 50)


,race,gender,age,time_in_hospital,A1Cresult,metformin,readmitted
0,Caucasian,Female,[0-10),1,NaN,No,NO
1,Caucasian,Female,[10-20),3,NaN,No,>30
2,AfricanAmerican,Female,[20-30),2,NaN,No,NO
3,Caucasian,Male,[30-40),2,NaN,No,NO


### Preprocessing steps
Everything lives in `fhg_examples/preprocessing.py::preprocess_diabetes`; the cells below run each step in isolation.

**Step 1 - drop unknowns.** Rows with unknown `race` ('?') or `gender` ('Unknown/Invalid') cannot be assigned to a group, so they are removed.

In [3]:
if HAS_RAW:
    known = raw[(raw.race != "?") & (raw.gender != "Unknown/Invalid")]
    print(f"{len(raw):,} -> {len(known):,} rows")

101,766 -> 99,492 rows


**Step 2 - label and sensitive attributes.** `readmitted` becomes 1 only for '<30' (the alternatives are '>30' and 'NO'). The three sensitive columns become 0/1 as described in the table below.

In [4]:
if HAS_RAW:
    step2 = pd.DataFrame({
        "race":   (known.race == "Caucasian").astype(int),
        "gender": (known.gender == "Male").astype(int),
        "age":    known.age.isin(pp.YOUNG_AGE_BINS).astype(int),
        "readmitted": (known.readmitted == "<30").astype(int)})
    display(step2.head(4))

,race,gender,age,readmitted
0,1,0,1,0
1,1,0,1,0
2,0,0,1,0
3,1,1,0,0


**Step 3 - encode clinical measurements as ordinal codes.** Lab results: Norm=0, elevated=1, very elevated=2, *not measured*=3 (missingness is kept as its own informative category). The 21 medication columns: Steady=0, Up=1, Down=2, No=3. `change` and `diabetesMed` become 0/1.

In [5]:
print(pp.LAB_RESULT_CODES, "| not measured ->", pp.LAB_NOT_MEASURED)
print(pp.MEDICATION_CODES)
if HAS_RAW:
    display(pd.crosstab(known.A1Cresult.fillna("not measured"), known.A1Cresult.map(pp.LAB_RESULT_CODES).fillna(3).astype(int)))

{'Norm': 0, '>200': 1, '>7': 1, '>300': 2, '>8': 2} | not measured -> 3
{'Steady': 0, 'Up': 1, 'Down': 2, 'No': 3}


A1Cresult,0,1,2,3
A1Cresult,,,,
>7,0,3730,0,0
>8,0,0,7961,0
Norm,4905,0,0,0
not measured,0,0,0,82896


**Step 4 - group diagnoses.** `diag_1..3` hold hundreds of ICD-9 codes. They are mapped to 9 coarse disease groups (Strack et al., 2014) and one-hot encoded. The identifiers `encounter_id`, `patient_nbr`, the mostly-missing `weight`, `payer_code`, `medical_specialty`, and the constant `examide`, `citoglipton` are dropped.

In [6]:
for c in ["250.83", "428", "486", "V57", "?"]:
    print(f"{c:>7} -> {pp.icd9_group(c)}")

 250.83 -> Diabetes
    428 -> Circulatory
    486 -> Respiratory
    V57 -> Other
      ? -> Other


### Verification

In [7]:
if HAS_RAW:
    rebuilt = pp.preprocess_diabetes(raw)
    assert rebuilt.equals(load("diabetes")), "preprocessing differs from the shipped CSV"
    print(f"✓ preprocess_diabetes(raw) reproduces the shipped CSV exactly {rebuilt.shape}")
else:
    print("Skipped (no raw file). The shipped CSV has shape", load("diabetes").shape)

✓ preprocess_diabetes(raw) reproduces the shipped CSV exactly (99492, 67)


### Sensitive attributes

In [8]:
diabetes = load("diabetes")
display(group_summary(diabetes, spec).round(3))

,attribute,value,group,n,share,base_rate
0,age,1,"privileged: younger than 30 (raw bins [0-10), ...",2453,0.025,0.112
1,age,0,unprivileged: 30 or older,97039,0.975,0.112
2,gender,1,privileged: Male,45917,0.462,0.111
3,gender,0,unprivileged: Female,53575,0.538,0.113
4,race,1,privileged: Caucasian,76099,0.765,0.113
5,race,0,"unprivileged: African American, Hispanic, Asia...",23393,0.235,0.110


### Why these groups?

- **race** - `1` = Caucasian, `0` = African American, Hispanic, Asian or Other (pooled). Racial and ethnic disparities in US healthcare access and quality are well documented (for example Obermeyer et al., *Science* 2019, found an algorithm used to allocate care under-served Black patients). Using the largest group as the reference follows that literature. Pooling the others is a simplification that hides differences between them.
- **gender** - `1` = Male, `0` = Female. A protected attribute in most anti-discrimination frameworks; the repository does not record why Male is the reference.
- **age** - `1` = younger than 30, `0` = 30 or older. **The reason for this choice is not recorded.** Note the group is tiny (about 2.5% of rows) and the base rates in the table are almost identical across *all* three attributes: here the interesting question is not a raw label gap but whether a model's *predictions* differ by group.

### Caveats about the diabetes data
- 99,492 encounters belong to only 69,667 unique patients (16,483 patients have more than one encounter), so the same patient can end up in both a training and a test split.
- 2,384 encounters end in death or hospice (discharge codes 11, 13, 14, 19, 20, 21). They are kept although those patients cannot be readmitted.

In [9]:
if HAS_RAW:
    print(f"{known.encounter_id.nunique():,} encounters, {known.patient_nbr.nunique():,} patients")
    print("death/hospice encounters:", known.discharge_disposition_id.isin([11, 13, 14, 19, 20, 21]).sum())

99,492 encounters, 69,667 patients
death/hospice encounters: 2384


## 2. Stroke prediction

*Source:* Kaggle `fedesoriano/stroke-prediction-dataset` (5,110 patients; see the dataset page for its license). *Label:* `stroke` = 1 if the patient had a stroke (a positive prediction is the favorable outcome: "flagged for follow-up").

### What was done before, and why it changed
The first version of this example (from the original research notebooks) had three problems, all fixed in the pipeline below:
1. **Resampled evaluation data.** The data was split 70/15/15 and only the training part was undersampled (NearMiss), then all parts were merged into one file. That raised the stroke rate from the true 4.87% to 13.2% and put NearMiss-selected rows into the validation/test sets of any later re-split.
2. **Label-tuned group definition.** The age cut-off (73) was found by searching thresholds on the labelled data. A group definition that depends on the outcome being audited is circular: it can make a disparity look larger, or smaller, by construction.
3. **Integer codes for nominal variables.** `work_type` and `smoking_status` were mapped to 0, 1, 2, ... which tells a model that categories are ordered and equally spaced.

In [10]:
spec_s = SPECS["stroke"]
HAS_RAW_S = raw_available("stroke")
if HAS_RAW_S:
    raw_s = load_raw("stroke")
    print(raw_s.shape)
    display(raw_s[["gender", "age", "ever_married", "bmi", "smoking_status", "stroke"]].head(4))
else:
    print("Raw file missing: see examples/data/raw/README.md. Showing the shipped CSV instead.")
    display(load("stroke").iloc[:4, :8])

(5110, 12)


,gender,age,ever_married,bmi,smoking_status,stroke
0,Male,67.0,Yes,36.6,formerly smoked,1
1,Female,61.0,Yes,NaN,never smoked,1
2,Male,80.0,Yes,32.5,never smoked,1
3,Female,49.0,Yes,34.4,smokes,1


### Corrected preprocessing (`preprocess_stroke`)

**Step 1 - clean.** Drop `id` and the single `gender == 'Other'` row (5,110 -> 5,109 rows).

**Step 2 - impute bmi.** The 201 missing `bmi` values get the mean bmi of patients with the same gender, marital status and age band (contiguous bands: [0,2], (2,12], (12,18], (18,35], (35,60], >60; the original code had a gap at (18,19]). The group means use all rows, i.e. they are computed *before* any split, which leaks a little information. We accept this for simplicity.

In [11]:
if HAS_RAW_S:
    clean = raw_s[raw_s.gender != "Other"].drop(columns="id")
    band = pd.cut(clean.age, pp.AGE_BAND_EDGES, labels=pp.AGE_BAND_NAMES)
    print("missing bmi:", clean.bmi.isna().sum())
    print(clean.groupby(band, observed=True).bmi.agg(["size", "mean"]).round(1).T)

missing bmi: 201
age   Infant  Child  Adolescent  Young  Adults     Old
size   175.0  413.0       328.0  982.0  1907.0  1304.0
mean    18.7   19.7        25.9   29.1    31.6    29.9


**Step 3 - sensitive attributes.** All binary, `1` = privileged (reference): `age` **65 or older** (the conventional older-adult threshold, also the US Medicare age; fixed *a priori*, never tuned on the labels), `gender` Male, `ever_married` Yes, `Residence_type` Urban.

**Step 4 - one-hot nominal variables.** `work_type` and `smoking_status` become one 0/1 column per level (no level dropped, for readability). `hypertension`, `heart_disease` are already 0/1.

**Step 5 - no resampling.** The file keeps the true 4.87% stroke rate in the original row order. Class imbalance is handled at *training* time (for example `BiasSearch(treat_umbalance=True)` or choosing the decision threshold by balanced accuracy), never in the data that is used for evaluation.

In [12]:
if HAS_RAW_S:
    out = pp.preprocess_stroke(raw_s)
    print(out.shape, f"stroke rate {out.stroke.mean():.4f}")
    display(out.filter(regex="^work_type|^smoking").head(3))

(5109, 18) stroke rate 0.0487


,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes
0,0,0,1,0,0,0,1,0,0
1,0,0,0,1,0,0,0,1,0
2,0,0,1,0,0,0,0,1,0


For comparison, the old resampling step (kept as `legacy_stroke_sampling`) applied to the same table:

In [13]:
if HAS_RAW_S:
    numeric = out.drop(columns=out.filter(regex="^work_type|^smoking").columns)
    old = pp.legacy_stroke_sampling(numeric)
    print(f"stroke rate: corrected {out.stroke.mean():.3f} | legacy {old.stroke.mean():.3f}")

stroke rate: corrected 0.049 | legacy 0.132


### Verification

In [14]:
if HAS_RAW_S:
    assert pp.matches_processed("stroke"), "preprocessing differs from the shipped CSV"
    print("✓ preprocess_stroke(raw) reproduces the shipped CSV (bmi compared to 6 decimals)")
else:
    print("Skipped (no raw file). The shipped CSV has shape", load("stroke").shape)

✓ preprocess_stroke(raw) reproduces the shipped CSV (bmi compared to 6 decimals)


### Sensitive attributes

In [15]:
display(group_summary(load("stroke"), spec_s).round(3))

,attribute,value,group,n,share,base_rate
0,age,1,privileged: 65 or older,1027,0.201,0.155
1,age,0,unprivileged: younger than 65,4082,0.799,0.022
2,gender,1,privileged: Male,2115,0.414,0.051
3,gender,0,unprivileged: Female,2994,0.586,0.047
4,ever_married,1,privileged: Yes (ever married),3353,0.656,0.066
5,ever_married,0,unprivileged: No,1756,0.344,0.017
6,Residence_type,1,privileged: Urban,2596,0.508,0.052
7,Residence_type,0,unprivileged: Rural,2513,0.492,0.045


### Why these groups?
- **age** - `1` = 65 or older. A conventional, a-priori threshold. Older patients are a small group (about 20%) with a much higher stroke rate, so a raw base-rate gap is expected here: it reflects real medical risk, so interpret "fairness" for this attribute carefully (equal outcomes are not the goal, equal *treatment of equally at-risk patients* is).
- **gender** - `1` = Male, `0` = Female. Protected attribute; the stroke rates are almost equal.
- **ever_married** - `1` = Yes. A proxy attribute strongly linked to age (married patients are older); the reason for its use is not recorded.
- **Residence_type** - `1` = Urban, `0` = Rural. Urban/rural is a commonly studied access-to-care dimension; the study's own reasoning is not recorded.

## 3. WiDS: metastatic cancer diagnosis

*Source:* Kaggle competition `widsdatathon2024-challenge1` (`training.csv`; competition data cannot be redistributed, so it is not in this repository). *Label:* `DiagPeriodL90D` = 1 if metastatic cancer was diagnosed within 90 days of screening.

The raw file is not available here, so only the shipped CSV is described. What the original workshop code documents: rows with more than 2 missing values were dropped; `patient_id`, `patient_gender`, `bmi` and the `metastatic_first_novel_treatment(_type)` columns were dropped; **`patient_age`** is 1 ('Younger', under 29) or 0 ('Older', 29 and over). **The definitions of `race`, `payer`, `state_privileged` and `neighborhood_education` are not recorded in the available code**: we only know they are 0/1 columns where, by callmefair's convention, 1 is the privileged group. Not knowing the cut-offs, we cannot explain their social rationale; the table gives the sizes and base rates so you can judge them.

In [16]:
wids = load("wids")
print(wids.shape)
display(group_summary(wids, SPECS["wids"]).drop(columns="group").round(3))

(12823, 21)


,attribute,value,n,share,base_rate
0,patient_age,1,105,0.008,0.771
1,patient_age,0,12718,0.992,0.623
2,race,1,9913,0.773,0.625
3,race,0,2910,0.227,0.624
4,payer,1,10279,0.802,0.628
5,payer,0,2544,0.198,0.611
6,state_privileged,1,4662,0.364,0.643
7,state_privileged,0,8161,0.636,0.614
8,neighborhood_education,1,6377,0.497,0.636
9,neighborhood_education,0,6446,0.503,0.614


## How callmefair sees the groups

callmefair consumes a standard AIF360 `BinaryLabelDataset` plus the group definitions. `spec.attribute(name).groups` returns them in the exact format AIF360 expects.

In [17]:
from aif360.datasets import BinaryLabelDataset

attr = spec.attribute("race")
privileged, unprivileged = attr.groups
ds = BinaryLabelDataset(df=diabetes[["race", "time_in_hospital", "readmitted"]], label_names=["readmitted"],
                        protected_attribute_names=["race"], favorable_label=1, unfavorable_label=0)
print("privileged_groups  =", privileged)
print("unprivileged_groups =", unprivileged)
print(ds.favorable_label, ds.protected_attribute_names, ds.features.shape)

privileged_groups  = [{'race': 1}]
unprivileged_groups = [{'race': 0}]
1.0 ['race'] (99492, 2)


### Base rate by group, all datasets
Each bar is the share of favorable labels (`1`) inside one group, before any model is trained, with the group size printed above it. Where the privileged and unprivileged bars differ (for example `age` in the stroke data), even a perfect model would give the groups different positive rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from callmefair.viz._style import new_axes, color, INK_SECONDARY

fig, axes = plt.subplots(1, len(SPECS), figsize=(15, 4.2), gridspec_kw={'width_ratios': [len(s.sensitive) + 0.8 for s in SPECS.values()]})
for ax, (name, s) in zip(axes, SPECS.items()):
    new_axes(ax)
    gs = group_summary(load(name), s)
    for k, (value, label) in enumerate([(1, 'privileged'), (0, 'unprivileged')]):
        sub = gs[gs.value == value].reset_index(drop=True)
        x = np.arange(len(sub)) + (k - 0.5) * 0.38
        ax.bar(x, sub.base_rate, 0.36, color=color(k), label=label, zorder=3)
        for xi, r, n in zip(x, sub.base_rate, sub.n):
            ax.text(xi, r, f'n={n:,}', ha='center', va='bottom', fontsize=7, color=INK_SECONDARY, rotation=90)
    ax.set_xticks(range(len(s.sensitive)), [a.name for a in s.sensitive], rotation=20, ha='right')
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
    ax.grid(False, axis='x')
    ax.set_title(f'{name}: P({s.label} = 1)')
axes[0].set_ylabel('label base rate')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
fig.tight_layout()
fig

### Summary of the datasets

In [18]:
rows = []
for name, s in SPECS.items():
    df = load(name)
    rows.append({"dataset": name, "rows": len(df), "features": df.shape[1] - 1, "label": s.label,
                 "base_rate": df[s.label].mean(), "sensitive": ", ".join(s.sensitive_names)})
display(pd.DataFrame(rows).round(3))

,dataset,rows,features,label,base_rate,sensitive
0,diabetes,99492,66,readmitted,0.112,"age, gender, race"
1,stroke,5109,17,stroke,0.049,"age, gender, ever_married, Residence_type"
2,wids,12823,20,DiagPeriodL90D,0.625,"patient_age, race, payer, state_privileged, ne..."


**Next:** [01_fairness_fundamentals.ipynb](01_fairness_fundamentals.ipynb) trains a baseline model on these tables and measures fairness for exactly these groups.